In [1]:
import pandas as pd

df = pd.read_csv(
    "../raw/Vittime.stato_civile.csv",
    sep=",",
    quotechar="'",
    encoding="utf-8-sig"
)

df.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,SEX,Sesso,MARITAL_STATUS,Stato civile,TIME_PERIOD,Osservazione,OBS_STATUS,Stato_osservazione
0,A,Annuale,IT,Italia,VICTIMS,Vittime di violenza,M,Maschi,SIN,Celibe/nubile,2015,130,NaN,NaN
1,A,Annuale,IT,Italia,VICTIMS,Vittime di violenza,M,Maschi,SIN,Celibe/nubile,2016,141,NaN,NaN
2,A,Annuale,IT,Italia,VICTIMS,Vittime di violenza,M,Maschi,SIN,Celibe/nubile,2017,129,NaN,NaN
3,A,Annuale,IT,Italia,VICTIMS,Vittime di violenza,M,Maschi,SIN,Celibe/nubile,2018,109,NaN,NaN
4,A,Annuale,IT,Italia,VICTIMS,Vittime di violenza,M,Maschi,SIN,Celibe/nubile,2019,93,NaN,NaN


In [2]:
# Per ogni colonna: numero di valori unici + elenco valori

for col in df.columns:
    print("=" * 60)
    print(f"Colonna: {col}")
    print(f"Numero valori unici: {df[col].nunique(dropna=False)}")
    print("Valori unici:")
    print(df[col].unique())
    print("\n")

Colonna: FREQ
Numero valori unici: 2
Valori unici:
['A' '"A']


Colonna: Frequenza
Numero valori unici: 1
Valori unici:
['Annuale']


Colonna: REF_AREA
Numero valori unici: 29
Valori unici:
['IT' 'ITC' 'ITC1' 'ITC2' 'ITC3' 'ITC4' 'ITD' 'ITDA' 'ITD1' 'ITD2' 'ITD3'
 'ITD4' 'ITD5' 'ITE' 'ITE1' 'ITE2' 'ITE3' 'ITE4' 'ITF' 'ITF1' 'ITF2'
 'ITF3' 'ITF4' 'ITF5' 'ITF6' 'ITG' 'ITG1' 'ITG2' 'ITNI']


Colonna: Territorio
Numero valori unici: 29
Valori unici:
['Italia' 'Nord-ovest' 'Piemonte' 'Valle d""Aosta / Vallée d""\'Aoste\''
 'Liguria' 'Lombardia' 'Nord-est' 'Trentino Alto Adige / Südtirol'
 'Provincia Autonoma Bolzano / Bozen' 'Provincia Autonoma Trento' 'Veneto'
 'Friuli-Venezia Giulia' 'Emilia-Romagna' 'Centro' 'Toscana' 'Umbria'
 'Marche' 'Lazio' 'Sud' 'Abruzzo' 'Molise' 'Campania' 'Puglia'
 'Basilicata' 'Calabria' 'Isole' 'Sicilia' 'Sardegna' 'Non indicato']


Colonna: DATA_TYPE
Numero valori unici: 1
Valori unici:
['VICTIMS']


Colonna: Indicatore
Numero valori unici: 1
Valori unici:
['V

In [3]:
# Rimozione colonne non informative o ridondanti:
# - colonne costanti (un solo valore per tutto il dataset)
# - colonne duplicate sul piano informativo
# - colonne completamente vuote

cols_to_drop = [
    "FREQ",               
    "Frequenza",          
    "REF_AREA",          
    "DATA_TYPE",          
    "Indicatore",         
    "SEX",                
    "MARITAL_STATUS",     
    "OBS_STATUS",         
    "Stato_osservazione" 
]

df = df.drop(columns=cols_to_drop)

# Verifica struttura finale
print("Nuova shape:", df.shape)
print("Colonne rimanenti:", df.columns.tolist())


Nuova shape: (5792, 5)
Colonne rimanenti: ['Territorio', 'Sesso', 'Stato civile', 'TIME_PERIOD', 'Osservazione']


In [4]:
# Standardizzazione nomi colonne:
# - tutto minuscolo
# - nomi più chiari e coerenti

df = df.rename(columns={
    "Territorio": "territorio",
    "Sesso": "sesso",
    "Stato civile": "stato_civile",
    "TIME_PERIOD": "periodo",
    "Osservazione": "numero_vittime"
})


# Eliminazione delle righe completamente vuote 
df = df.dropna(how="all")

# Stampa delle colonne e dei valori unici che contengono
for col in df.columns:
    print("\nColonna:", col)
    print(df[col].unique())



Colonna: territorio
['Italia' 'Nord-ovest' 'Piemonte' 'Valle d""Aosta / Vallée d""\'Aoste\''
 'Liguria' 'Lombardia' 'Nord-est' 'Trentino Alto Adige / Südtirol'
 'Provincia Autonoma Bolzano / Bozen' 'Provincia Autonoma Trento' 'Veneto'
 'Friuli-Venezia Giulia' 'Emilia-Romagna' 'Centro' 'Toscana' 'Umbria'
 'Marche' 'Lazio' 'Sud' 'Abruzzo' 'Molise' 'Campania' 'Puglia'
 'Basilicata' 'Calabria' 'Isole' 'Sicilia' 'Sardegna' 'Non indicato']

Colonna: sesso
['Maschi' 'Femmine' 'Non indicato' 'Totale']

Colonna: stato_civile
['Celibe/nubile' 'Separato di fatto/legalmente da matrimonio'
 'Coniugato, unito civilmente, separato da matrimonio/unione civile'
 'Divorziato da matrimonio/unione civile'
 'Vedovo da matrimonio/unione civile' 'Non indicato' 'Non disponibile'
 'Totale']

Colonna: periodo
[2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]

Colonna: numero_vittime
[  130   141   129   109    93   250   263   208   243    40    24    30
    18    10    28    25    14    27    90    98    66

In [5]:
# Pulizia colonna territorio:
# - uniformazione minuscolo
# - correzione denominazioni incoerenti / verbose

df["territorio"] = (
    df["territorio"]
    .str.lower()
    .str.strip()
    .str.replace("-", " ", regex=False)
)


df["territorio"] = df["territorio"].replace({
    'valle d""aosta / vallée d""\'aoste\'': "valle d'aosta",
    "trentino alto adige / südtirol": "trentino alto adige",
    "provincia autonoma bolzano / bozen": "bolzano",
    "provincia autonoma trento": "trento"
})


# Uniformazione minuscolo per la colonna sesso e stato_civile
df["sesso"] = df["sesso"].str.lower().str.strip()
df["stato_civile"] = df["stato_civile"].str.lower().str.strip()


# Rimozione della voce aggregata "totale" 
# da sesso e stato_civile per evitare doppio conteggio

df = df[
    (df["sesso"] != "totale") &
    (df["stato_civile"] != "totale")
]


# Modifica denominazioni verbose e non inclusive per la colonna stato_civile
df["stato_civile"] = df["stato_civile"].replace({
    "celibe/nubile": "single",
    "separato di fatto/legalmente da matrimonio": "separat",
    "coniugato, unito civilmente, separato da matrimonio/unione civile": "coniugat",
    "divorziato da matrimonio/unione civile": "divorziat",
    "vedovo da matrimonio/unione civile": "vedov",
    "non indicato": "non indicato",
    "non disponibile": "non disponibile"
})


# Conversione delle colonne numeriche da float a integer

df["periodo"] = df["periodo"].astype(int)
df["numero_vittime"] = df["numero_vittime"].astype(int)


# Verifica finale post-pulizia (struttura + qualità base)

# 1) Dimensioni e info colonne
display(df.head(10))
df.info()

# 2) Tipi e missing
print("\nMissing values per colonna:")
display(df.isna().sum().sort_values(ascending=False))

# 3) Valori unici (conteggio) per colonna
print("\nNumero valori unici per colonna:")
display(df.nunique(dropna=False).sort_values(ascending=False))

# 4) Check duplicati (righe identiche)
print("\nRighe duplicate (identiche su tutte le colonne):", df.duplicated().sum())

# 5) Valori distinti per ogni colonna
for col in df.columns:
    print("\nColonna:", col)
    
    if col == "periodo":
        print(sorted(df[col].astype(int).unique().tolist()))
    else:
        print(df[col].unique())

,territorio,sesso,stato_civile,periodo,numero_vittime
0,italia,maschi,single,2015,130
1,italia,maschi,single,2016,141
2,italia,maschi,single,2017,129
3,italia,maschi,single,2018,109
4,italia,maschi,single,2019,93
5,italia,maschi,single,2020,250
6,italia,maschi,single,2021,263
7,italia,maschi,single,2022,141
8,italia,maschi,single,2023,208
9,italia,maschi,single,2024,243


<class 'pandas.core.frame.DataFrame'>
Index: 3062 entries, 0 to 5704
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   territorio      3062 non-null   object
 1   sesso           3062 non-null   object
 2   stato_civile    3062 non-null   object
 3   periodo         3062 non-null   int64 
 4   numero_vittime  3062 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 143.5+ KB

Missing values per colonna:


territorio        0
sesso             0
stato_civile      0
periodo           0
numero_vittime    0
dtype: int64


Numero valori unici per colonna:


numero_vittime    476
territorio         29
periodo            10
stato_civile        7
sesso               3
dtype: int64


Righe duplicate (identiche su tutte le colonne): 0

Colonna: territorio
['italia' 'nord ovest' 'piemonte' "valle d'aosta" 'liguria' 'lombardia'
 'nord est' 'trentino alto adige' 'bolzano' 'trento' 'veneto'
 'friuli venezia giulia' 'emilia romagna' 'centro' 'toscana' 'umbria'
 'marche' 'lazio' 'sud' 'abruzzo' 'molise' 'campania' 'puglia'
 'basilicata' 'calabria' 'isole' 'sicilia' 'sardegna' 'non indicato']

Colonna: sesso
['maschi' 'femmine' 'non indicato']

Colonna: stato_civile
['single' 'separat' 'coniugat' 'divorziat' 'vedov' 'non indicato'
 'non disponibile']

Colonna: periodo
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Colonna: numero_vittime
[ 130  141  129  109   93  250  263  208  243   40   24   30   18   10
   28   25   14   27   90   98   66   74  145  103   44  115  112    6
   12    9   13    8    7    5    4   11    2   15    3    1   83   26
   58   32   33   21   41   54 2235 2364 2449 2918 2977 5318 6157 4630
 6261 7394  824  849  822  926  843 1217 

In [6]:
# Salvataggio dataset pulito

output_path = "../data_clean/vittime.stato_civile.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("File salvato in:", output_path)

File salvato in: ../data_clean/vittime.stato_civile.csv
